# Current GLM-OCR checkpoint: beam 4 plus character KenLM

This inference-only notebook tests whether character-level KenLM shallow fusion adds value **after ordinary beam 4 produced almost no gain**. It uses the current best checkpoint (`stage2/checkpoints/checkpoint-642`), the current fixed 400-image holdout, the native OCR prompt and the same spatial preprocessing and stopping tokens as training.

The protocol is intentionally separated:

1. Train a character 6-gram only on the current **training labels**.
2. Use a deterministic 100-image calibration subset to choose among no fusion and λ = 0.025, 0.05, 0.10.
3. Freeze that choice and evaluate once on the other 300 images.
4. Compare it with beam 4 and greedy predictions already generated by `glm_greedy_vs_beam4.ipynb`.

Predictions are cached after every image and configuration. Rerunning safely resumes. This notebook does not train GLM-OCR, edit either training notebook, overwrite `final_selection.json`, or create a submission.

The checkpoint was already selected using all 400 validation images, so the 300-image result is useful decoder evidence but not a completely untouched model test. The calibration/evaluation separation only prevents KenLM λ from being selected and judged on the same rows.

In [ ]:
from __future__ import annotations
import gc, hashlib, json, math, os, platform, shutil, subprocess, sys, time, unicodedata
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

try:
    import kenlm
except ImportError as error:
    raise ImportError(
        'The Python package `kenlm` is required in this notebook kernel. Install/build KenLM '
        'in the same environment, restart the kernel, and rerun. The older project KenLM '
        'notebook may already have a working Linux environment.'
    ) from error

import numpy as np
import pandas as pd
import torch, transformers, peft
from PIL import Image, ImageOps
from peft import PeftModel
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from transformers import AutoProcessor, GlmOcrForConditionalGeneration, LogitsProcessor, LogitsProcessorList
from IPython.display import display

ROOT = Path.cwd().resolve()
assert (ROOT / 'resources/Train.csv').is_file(), 'Open this notebook in the project root.'
RUN = ROOT / 'training_outputs/glm_ocr_from_base_curriculum_v7_native_ocr_balanced'
BASE_MODEL = ROOT / 'model/glm-ocr'
IMAGES = ROOT / 'resources/images'
NOTEBOOK_PATH = ROOT / 'notebooks/glm_kenlm_decoding.ipynb'

NGRAM_ORDER = 6
FUSION_LAMBDAS = (0.025, 0.05, 0.10)
NUM_BEAMS = 4
LENGTH_PENALTY = 1.0
CANDIDATE_TOP_K = 256
CALIBRATION_SIZE = 100
CALIBRATION_SEED = 314159
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 271828

run_manifest = json.loads((RUN / 'run_identity.json').read_text(encoding='utf-8'))
CONFIG = run_manifest['identity']['config']
selection = json.loads((RUN / 'final_selection.json').read_text(encoding='utf-8'))
SPEC = selection['spec']
MODEL_PATH = RUN / SPEC['path']
assert SPEC == {'kind':'adapter','path':'stage2/checkpoints/checkpoint-642'}, SPEC
assert (MODEL_PATH / 'adapter_model.safetensors').is_file()
assert CONFIG['prompt'] == 'Text Recognition:'

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
DEVICE = 'cuda:0'
normalize = lambda value: unicodedata.normalize('NFC', str(value)).strip()
pd.set_option('display.max_colwidth', 180)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

## 1. Verify the split and import the completed greedy/beam baseline

The baseline is located by its experiment metadata rather than a hard-coded hash directory. It must use checkpoint 642, beam sizes 1 and 4, the same validation IDs, and predictions that independently reproduce its saved scores. No baseline prediction is regenerated here.

In [ ]:
audit = pd.read_csv(RUN / 'split_and_label_audit.csv', dtype={'ID':str}, keep_default_na=False)
validation = pd.read_csv(RUN / 'validation_manifest.csv', dtype={'ID':str}, keep_default_na=False).sort_values('ID').reset_index(drop=True)
train = audit.loc[audit.split.eq('train')].copy().sort_values('ID').reset_index(drop=True)
assert len(validation) == 400 and validation.ID.is_unique and train.ID.is_unique
assert set(validation.ID) == set(audit.loc[audit.split.eq('validation'),'ID'])
assert set(validation.ID).isdisjoint(train.ID)
assert set(validation.ID).isdisjoint(run_manifest['identity']['excluded_ids'])

beam_roots = []
for result_path in (RUN / 'greedy_vs_beam4').glob('*/comparison_result.json'):
    result = json.loads(result_path.read_text(encoding='utf-8'))
    experiment = json.loads((result_path.parent / 'experiment.json').read_text(encoding='utf-8'))
    if result['best_checkpoint'] == SPEC and set(map(int, experiment['generation'])) == {1,4}:
        beam_roots.append(result_path.parent)
assert len(beam_roots) == 1, f'Expected one compatible completed beam comparison, found: {beam_roots}'
BEAM_RUN = beam_roots[0]
baseline_summary = pd.read_csv(BEAM_RUN / 'summary.csv').set_index('num_beams')
greedy = pd.read_csv(BEAM_RUN / 'beam1_predictions.csv', dtype={'ID':str}, keep_default_na=False)
beam4 = pd.read_csv(BEAM_RUN / 'beam4_predictions.csv', dtype={'ID':str}, keep_default_na=False)
for frame in (greedy, beam4):
    assert frame.ID.is_unique and set(frame.ID) == set(validation.ID)
    assert frame.set_index('ID').Target.map(normalize).sort_index().equals(validation.set_index('ID').Target.map(normalize).sort_index())
print('Training labels for KenLM:', len(train))
print('Compatible baseline:', BEAM_RUN)
display(baseline_summary)

## 2. Build a validation-safe character 6-gram

Each Unicode character becomes a token such as `U000061`; this preserves spaces, carets, punctuation, capitalization and historical spelling. `lmplz` estimates a modified Kneser–Ney model. The corpus contains only current training labels—never calibration/evaluation labels or predictions.

Set `KENLM_BIN_DIR` below if `lmplz` and optional `build_binary` are not on PATH. The old project commonly used `.kenlm_wsl_build/bin`. Loading ARPA directly works when `build_binary` is unavailable, though a binary is faster and smaller.

In [ ]:
# Edit only this path if your KenLM command-line tools are elsewhere.
KENLM_BIN_DIR = ROOT / '.kenlm_wsl_build/bin'

def find_tool(name):
    candidates = [shutil.which(name), KENLM_BIN_DIR / name, KENLM_BIN_DIR / f'{name}.exe']
    return next((Path(path) for path in candidates if path and Path(path).is_file()), None)

LMPLZ = find_tool('lmplz')
BUILD_BINARY = find_tool('build_binary')
if LMPLZ is None:
    raise FileNotFoundError(
        'Cannot find the KenLM `lmplz` executable. Build/install KenLM, then either put '
        '`lmplz` on PATH or set KENLM_BIN_DIR above. This notebook will not download or '
        'silently install a different toolchain.'
    )

LM_ROOT = RUN / 'current_checkpoint_kenlm' / 'train_only_char6'
LM_ROOT.mkdir(parents=True, exist_ok=True)
LM_CORPUS = LM_ROOT / 'train_labels_characters.txt'
LM_ARPA = LM_ROOT / 'train_labels_char6.arpa'
LM_BINARY = LM_ROOT / 'train_labels_char6.binary'
LM_METADATA = LM_ROOT / 'metadata.json'

def character_tokens(text):
    return ' '.join(f'U{ord(character):06X}' for character in normalize(text))

expected_lm_metadata = {
    'order':NGRAM_ORDER, 'rows':len(train),
    'ids_sha256':hashlib.sha256('\n'.join(train.ID).encode()).hexdigest(),
    'labels_sha256':hashlib.sha256('\n'.join(map(normalize,train.Target)).encode()).hexdigest(),
    'corpus_policy':'NFC + edge trim; one Uxxxxxx token per Unicode character',
}
metadata_matches = LM_METADATA.exists() and json.loads(LM_METADATA.read_text(encoding='utf-8')) == expected_lm_metadata
use_binary = LM_BINARY.exists() and metadata_matches
if not LM_ARPA.exists() or not metadata_matches:
    LM_CORPUS.write_text('\n'.join(map(character_tokens,train.Target))+'\n', encoding='utf-8')
    subprocess.run([str(LMPLZ),'-o',str(NGRAM_ORDER),'--discount_fallback','-S','25%',
                    '-T',str(LM_ROOT),'--text',str(LM_CORPUS),'--arpa',str(LM_ARPA)], check=True)
    if BUILD_BINARY:
        temporary = LM_BINARY.with_suffix('.binary.tmp')
        subprocess.run([str(BUILD_BINARY),'trie',str(LM_ARPA),str(temporary)], check=True)
        temporary.replace(LM_BINARY)
        use_binary = True
    else:
        use_binary = False
    write_json(LM_METADATA, expected_lm_metadata)

lm_path = LM_BINARY if use_binary else LM_ARPA
character_lm = kenlm.Model(str(lm_path))
assert character_lm.order == NGRAM_ORDER
print('Loaded:', lm_path)
print('Confirmed training-only rows:', len(train))

## 3. Freeze calibration and evaluation IDs

The split is deterministic and stratified by label group and crop-height band when strata permit it. Its manifests are written once and then checked exactly on reruns. Changing the seed or size requires a separate output directory.

In [ ]:
from sklearn.model_selection import train_test_split
strata = validation.label_group + '/' + validation.height_band
strata = strata.where(strata.map(strata.value_counts()) >= 2, validation.label_group)
calibration_ids, evaluation_ids = train_test_split(validation.ID, train_size=CALIBRATION_SIZE,
    random_state=CALIBRATION_SEED, shuffle=True, stratify=strata)
calibration = validation[validation.ID.isin(calibration_ids)].sort_values('ID').reset_index(drop=True)
evaluation = validation[validation.ID.isin(evaluation_ids)].sort_values('ID').reset_index(drop=True)
assert len(calibration)==100 and len(evaluation)==300 and set(calibration.ID).isdisjoint(evaluation.ID)
SPLIT_ROOT = RUN / 'current_checkpoint_kenlm' / f'cal{CALIBRATION_SIZE}_seed{CALIBRATION_SEED}'
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
for name, frame in [('calibration',calibration),('evaluation',evaluation)]:
    path=SPLIT_ROOT/f'{name}_manifest.csv'
    if path.exists():
        saved_frame=pd.read_csv(path,dtype={'ID':str},keep_default_na=False)
        assert saved_frame.equals(frame), f'{name} manifest changed; choose a new output directory.'
    else: frame.to_csv(path,index=False)
display(pd.crosstab(validation.label_group,
    np.where(validation.ID.isin(calibration.ID),'calibration','evaluation')))

## 4. Native GLM-OCR preprocessing and incremental shallow fusion

At each beam expansion, the combined score is:

`GLM natural-log score + λ × ln(10) × incremental KenLM log10 score`.

KenLM scores only the top 256 neural token candidates per beam for tractability, while always retaining stop tokens. This is approximate shallow fusion; a candidate outside the neural top 256 cannot be recovered by KenLM. No repetition penalty or sampling is added.

In [ ]:
processor = AutoProcessor.from_pretrained(BASE_MODEL, local_files_only=True, use_fast=False)
base_generation = json.loads((BASE_MODEL / 'generation_config.json').read_text(encoding='utf-8'))
base_eos = base_generation['eos_token_id']; base_eos = [base_eos] if isinstance(base_eos,int) else list(base_eos)
EOS_IDS = sorted(set(base_eos + [processor.tokenizer.eos_token_id]))

def resize_spatial(image):
    image=image.convert('RGB')
    for bound, comparison in [(CONFIG['max_spatial_pixels'],lambda area,x:area>x),
                              (CONFIG['min_spatial_pixels'],lambda area,x:area<x)]:
        if comparison(image.width*image.height,bound):
            scale=math.sqrt(bound/(image.width*image.height))
            image=image.resize((max(1,int(image.width*scale)),max(1,int(image.height*scale))),Image.Resampling.BICUBIC)
    return image

def encode_image(image_id):
    with Image.open(IMAGES/f'{image_id}.jpg') as source:
        image=resize_spatial(ImageOps.exif_transpose(source).convert('RGB'))
    text=('[gMASK]<sop><|user|>\n<|begin_of_image|><|image|><|end_of_image|>'+CONFIG['prompt']+'<|assistant|>')
    inputs=processor(text=[text],images=[image],add_special_tokens=False,return_tensors='pt')
    inputs.pop('token_type_ids',None)
    assert int(inputs['input_ids'][0,-1])==processor.tokenizer.convert_tokens_to_ids('<|assistant|>')
    assert 'mm_token_type_ids' in inputs
    return inputs

GLOBAL_PIECES = {}
class CharacterKenLMFusion(LogitsProcessor):
    def __init__(self, prompt_length, fusion_lambda):
        self.prompt_length=int(prompt_length);self.fusion_lambda=float(fusion_lambda)
        self.eos_ids=set(map(int,EOS_IDS));self.special_ids=set(processor.tokenizer.all_special_ids)
        self.states={};self.transitions={};self.ln10=math.log(10.0)
        state=kenlm.State();character_lm.BeginSentenceWrite(state);self.states[()]=state

    def piece(self,token_id):
        token_id=int(token_id)
        if token_id not in GLOBAL_PIECES:
            if token_id in self.special_ids:piece=None
            else:
                piece=unicodedata.normalize('NFC',processor.tokenizer.decode([token_id],
                    skip_special_tokens=False,clean_up_tokenization_spaces=False))
                if not piece or '\ufffd' in piece:piece=None
            GLOBAL_PIECES[token_id]=piece
        return GLOBAL_PIECES[token_id]

    def advance(self,state,text):
        total=0.0;current=state
        for character in text:
            output=kenlm.State();total+=character_lm.BaseScore(current,f'U{ord(character):06X}',output);current=output
        return total,current

    def state_for(self,prefix):
        if prefix not in self.states:
            parent=prefix[:-1];piece=self.piece(prefix[-1])
            if piece is None:raise RuntimeError(f'Unsupported active token: {prefix[-1]}')
            _,self.states[prefix]=self.advance(self.state_for(parent),piece)
        return self.states[prefix]

    def delta(self,prefix,token_id):
        key=(prefix,int(token_id))
        if key in self.transitions:return self.transitions[key]
        state=self.state_for(prefix)
        if token_id in self.eos_ids:
            output=kenlm.State();value=character_lm.BaseScore(state,'</s>',output)
        else:
            piece=self.piece(token_id)
            if piece is None:return None
            value,_=self.advance(state,piece)
        self.transitions[key]=float(value);return float(value)

    def __call__(self,input_ids,scores):
        k=min(CANDIDATE_TOP_K,scores.shape[-1])
        candidate_rows=torch.topk(scores,k=k,dim=-1).indices.tolist()
        fused=torch.full_like(scores,-torch.inf)
        for row_index,token_ids in enumerate(candidate_rows):
            prefix=tuple(map(int,input_ids[row_index,self.prompt_length:].detach().cpu().tolist()))
            candidates=set(token_ids)|self.eos_ids;valid=[];bonuses=[]
            for token_id in candidates:
                bonus=self.delta(prefix,token_id)
                if bonus is not None:valid.append(token_id);bonuses.append(bonus)
            index=torch.tensor(valid,device=scores.device,dtype=torch.long)
            fused[row_index,index]=scores[row_index,index]+self.fusion_lambda*self.ln10*torch.tensor(
                bonuses,device=scores.device,dtype=scores.dtype)
        return fused

## 5. Model, experiment identity and resumable inference

Cache identity covers model/LM contents, split manifests, notebook source, images, runtime versions and decoding parameters. Timing synchronizes CUDA around generation and excludes preprocessing and disk I/O. Warmup is excluded. If GPU memory is exhausted, completed predictions stay cached and settings are not silently changed.

In [ ]:
@dataclass(frozen=True)
class FusionConfig:
    name:str
    fusion_lambda:float
    @property
    def cache_key(self):return f'{self.name}_beam4_lambda{self.fusion_lambda:g}'.replace('.','p')

NO_FUSION=FusionConfig('visual',0.0)
FUSION_CONFIGS=[FusionConfig('fusion',value) for value in FUSION_LAMBDAS]

def load_model():
    assert torch.cuda.is_available(),'CUDA GPU required for shallow-fusion inference.'
    gc.collect();torch.cuda.empty_cache()
    base=GlmOcrForConditionalGeneration.from_pretrained(BASE_MODEL,local_files_only=True,dtype=DTYPE,
        device_map={'':0},attn_implementation='sdpa')
    model=PeftModel.from_pretrained(base,MODEL_PATH,is_trainable=False).eval()
    for parameter in model.parameters():parameter.requires_grad_(False)
    return model

def decode(model,inputs,config):
    prompt_length=int(inputs['input_ids'].shape[1]);processors=LogitsProcessorList()
    if config.fusion_lambda>0:processors.append(CharacterKenLMFusion(prompt_length,config.fusion_lambda))
    options={'max_new_tokens':CONFIG['max_new_tokens'],'num_beams':NUM_BEAMS,'num_return_sequences':1,
        'do_sample':False,'use_cache':True,'eos_token_id':EOS_IDS,'pad_token_id':processor.tokenizer.pad_token_id,
        'length_penalty':LENGTH_PENALTY,'early_stopping':True,'logits_processor':processors,'renormalize_logits':False}
    torch.cuda.synchronize();started=time.perf_counter()
    with torch.inference_mode():output=model.generate(**inputs,**options)
    torch.cuda.synchronize();elapsed=time.perf_counter()-started
    token_ids=output[0,prompt_length:].tolist()
    eos_index=next((i for i,token in enumerate(token_ids) if token in EOS_IDS),None)
    if eos_index is not None:token_ids=token_ids[:eos_index+1]
    prediction=normalize(processor.decode(token_ids,skip_special_tokens=True,clean_up_tokenization_spaces=False))
    return {'Prediction':prediction,'generated_tokens':len(token_ids),'stopped_with_eos':eos_index is not None,
            'hit_token_limit':eos_index is None and len(token_ids)>=CONFIG['max_new_tokens'],'seconds':elapsed}

notebook=json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
# The submission cell is deliberately excluded so it does not invalidate completed validation caches.
notebook_code=object_hash([''.join(c['source']) if c.get('id')!='7cb8114162a6bb79' else '' for c in notebook['cells'] if c['cell_type']=='code'])
image_hashes={image_id:file_hash(IMAGES/f'{image_id}.jpg') for image_id in tqdm(validation.ID,desc='Fingerprint images')}
identity={'checkpoint':SPEC,'checkpoint_files':{p.name:file_hash(p) for p in MODEL_PATH.iterdir()
    if p.suffix in {'.json','.safetensors'}},'base_files':{p.name:file_hash(p) for p in BASE_MODEL.iterdir()
    if p.suffix in {'.json','.safetensors','.jinja','.model'}},'lm_metadata':expected_lm_metadata,
    'lm_hash':file_hash(lm_path),'calibration_manifest':file_hash(SPLIT_ROOT/'calibration_manifest.csv'),
    'evaluation_manifest':file_hash(SPLIT_ROOT/'evaluation_manifest.csv'),'images':image_hashes,
    'code':notebook_code,'runtime':{'python':platform.python_version(),'torch':torch.__version__,
    'transformers':transformers.__version__,'peft':peft.__version__,'kenlm_order':character_lm.order,
    'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    'settings':{'beam':NUM_BEAMS,'length_penalty':LENGTH_PENALTY,'top_k':CANDIDATE_TOP_K,
    'lambdas':FUSION_LAMBDAS,'max_new_tokens':CONFIG['max_new_tokens'],'eos_ids':EOS_IDS}}
EXPERIMENT_KEY=object_hash(identity)
OUT=SPLIT_ROOT/EXPERIMENT_KEY[:16];OUT.mkdir(parents=True,exist_ok=True)
manifest=OUT/'experiment.json'
if manifest.exists():assert json.loads(manifest.read_text(encoding='utf-8'))==identity
else:write_json(manifest,identity)

def cache_path(split_name,config,image_id):return OUT/'cache'/split_name/config.cache_key/f'{image_id}.json'
def cached(split_name,config,image_id):
    path=cache_path(split_name,config,image_id)
    if not path.exists():return None
    row=json.loads(path.read_text(encoding='utf-8'))
    assert row['experiment_key']==EXPERIMENT_KEY and row['image_hash']==image_hashes[image_id]
    assert row['config']==asdict(config) and row['ID']==image_id
    return row

def run_configs(frame,split_name,configs):
    pending=[(row.ID,c) for row in frame.itertuples() for c in configs if cached(split_name,c,row.ID) is None]
    if not pending:print('All requested predictions already cached:',split_name);return
    model=load_model()
    try:
        warm=encode_image(frame.iloc[0].ID).to(DEVICE)
        for config in configs[:1]:decode(model,warm,config)
        del warm
        for row in tqdm(frame.itertuples(),total=len(frame),desc=split_name):
            needed=[c for c in configs if cached(split_name,c,row.ID) is None]
            if not needed:continue
            inputs=encode_image(row.ID).to(DEVICE)
            for config in needed:
                try:result=decode(model,inputs,config)
                except torch.cuda.OutOfMemoryError:
                    print('GPU memory exhausted. Free memory and rerun; cached work is preserved.')
                    raise
                write_json(cache_path(split_name,config,row.ID),{'experiment_key':EXPERIMENT_KEY,
                    'image_hash':image_hashes[row.ID],'ID':row.ID,'config':asdict(config),**result})
            del inputs
    finally:
        del model;gc.collect();torch.cuda.empty_cache()

print('Experiment output:',OUT)

## RUN 1 — calibrate λ on 100 images

This is the first long inference cell: three fused beam-4 settings × 100 images. The no-fusion baseline is imported from the completed beam notebook.

In [ ]:
run_configs(calibration,'calibration',FUSION_CONFIGS)

## 6. Select λ using calibration only

The frozen choice is the highest proxy. No fusion participates as a candidate; ties favor no fusion, then smaller λ. If no fusion wins, the evaluation section reuses baseline predictions and performs no unnecessary KenLM inference.

In [ ]:
def add_errors(frame):
    frame=frame.copy();frame['Target']=frame.Target.map(normalize);frame['Prediction']=frame.Prediction.map(normalize)
    frame['nwords']=frame.Target.str.split().str.len().clip(lower=1);frame['nchars']=frame.Target.str.len().clip(lower=1)
    frame['word_edits']=[Levenshtein.distance(a.split(),b.split()) for a,b in zip(frame.Target,frame.Prediction)]
    frame['char_edits']=[Levenshtein.distance(a,b) for a,b in zip(frame.Target,frame.Prediction)]
    return frame
def summarize(frame):
    ww,cw=np.sqrt(frame.nwords),np.sqrt(frame.nchars)
    ew=np.dot(frame.word_edits,ww)/ww.sum();ec=np.dot(frame.char_edits,cw)/cw.sum()
    return {'images':len(frame),'weighted_wer':float(ew),'weighted_cer':float(ec),
        'zindi_proxy':float(1-.5*(ew/CONFIG['word_normalizer']+ec/CONFIG['character_normalizer'])),
        'WER_percent':float(100*frame.word_edits.sum()/frame.nwords.sum()),
        'CER_percent':float(100*frame.char_edits.sum()/frame.nchars.sum()),
        'exact_lines':int(frame.Target.eq(frame.Prediction).sum()),
        'blank_outputs':int(frame.Prediction.eq('').sum()),
        'token_limit_hits':int(frame.get('hit_token_limit',pd.Series(False,index=frame.index)).astype(bool).sum()),
        'mean_seconds':float(frame.seconds.mean()) if 'seconds' in frame else None}
def load_fusion(frame,split_name,config):
    rows=[cached(split_name,config,image_id) for image_id in frame.ID]
    assert all(row is not None for row in rows),'Run the preceding inference cell.'
    return add_errors(frame.merge(pd.DataFrame(rows),on='ID',validate='one_to_one'))
def imported_baseline(frame):
    columns=['ID','Prediction','generated_tokens','hit_token_limit','generation_seconds']
    available=[c for c in columns if c in beam4]
    result=frame.merge(beam4[available],on='ID',validate='one_to_one')
    if 'generation_seconds' in result:result=result.rename(columns={'generation_seconds':'seconds'})
    return add_errors(result)

calibration_results={NO_FUSION.cache_key:imported_baseline(calibration)}
for config in FUSION_CONFIGS:calibration_results[config.cache_key]=load_fusion(calibration,'calibration',config)
calibration_table=pd.DataFrame([{'name':key,**summarize(frame)} for key,frame in calibration_results.items()])
calibration_table['fusion_lambda']=[0.0,*FUSION_LAMBDAS]
calibration_table=calibration_table.sort_values(['zindi_proxy','fusion_lambda'],ascending=[False,True]).reset_index(drop=True)
display(calibration_table);calibration_table.to_csv(OUT/'calibration_results.csv',index=False)
chosen_lambda=float(calibration_table.iloc[0].fusion_lambda)
CHOSEN=NO_FUSION if chosen_lambda==0 else next(c for c in FUSION_CONFIGS if c.fusion_lambda==chosen_lambda)
write_json(OUT/'frozen_choice.json',{'selected_on':'calibration only','config':asdict(CHOSEN),
    'calibration_metrics':calibration_table.iloc[0].to_dict()})
print('Frozen choice:',CHOSEN)

## RUN 2 — evaluate the frozen choice on the other 300 images

If calibration selected fusion, this is one fused beam-4 run over 300 images. If it selected no fusion, the cell reports that existing baseline predictions are sufficient.

In [ ]:
if CHOSEN.fusion_lambda > 0:run_configs(evaluation,'evaluation',[CHOSEN])
else:print('Calibration selected no fusion; no additional inference is necessary.')

## 7. Final paired comparison

The evaluation rows were not used to choose λ. A paired bootstrap resamples whole lines and recomputes both weighted proxy scores. Its interval describes sampling variability conditional on this checkpoint and split; it does not correct prior checkpoint selection on the broader holdout or writer/page dependence.

In [ ]:
baseline_eval=imported_baseline(evaluation)
chosen_eval=(baseline_eval.copy() if CHOSEN.fusion_lambda==0 else load_fusion(evaluation,'evaluation',CHOSEN))
summary=pd.DataFrame([{'decoder':'beam4','fusion_lambda':0.0,**summarize(baseline_eval)},
                      {'decoder':'selected','fusion_lambda':CHOSEN.fusion_lambda,**summarize(chosen_eval)}])
display(summary);summary.to_csv(OUT/'evaluation_summary.csv',index=False)

assert baseline_eval.ID.tolist()==chosen_eval.ID.tolist() and baseline_eval.Target.tolist()==chosen_eval.Target.tolist()
paired=baseline_eval[['ID','Target','Prediction','word_edits','char_edits']].rename(columns={
    'Prediction':'beam4_prediction','word_edits':'beam4_word_edits','char_edits':'beam4_char_edits'})
paired['kenlm_prediction']=chosen_eval.Prediction;paired['kenlm_word_edits']=chosen_eval.word_edits
paired['kenlm_char_edits']=chosen_eval.char_edits
print('KenLM fewer/equal/more character errors:',int((chosen_eval.char_edits<baseline_eval.char_edits).sum()),
      int((chosen_eval.char_edits==baseline_eval.char_edits).sum()),int((chosen_eval.char_edits>baseline_eval.char_edits).sum()))

def score_indices(frame,indices):
    word_weights=np.sqrt(frame.nwords.to_numpy()[indices]);char_weights=np.sqrt(frame.nchars.to_numpy()[indices])
    ew=(frame.word_edits.to_numpy()[indices]*word_weights).sum()/word_weights.sum()
    ec=(frame.char_edits.to_numpy()[indices]*char_weights).sum()/char_weights.sum()
    return 1-.5*(ew/CONFIG['word_normalizer']+ec/CONFIG['character_normalizer'])
rng=np.random.default_rng(BOOTSTRAP_SEED)
indices=rng.integers(0,len(evaluation),size=(BOOTSTRAP_REPLICATES,len(evaluation)))
deltas=np.array([score_indices(chosen_eval,index)-score_indices(baseline_eval,index) for index in indices])
point=summarize(chosen_eval)['zindi_proxy']-summarize(baseline_eval)['zindi_proxy']
low,high=np.quantile(deltas,[.025,.975])
print(f'Frozen KenLM minus beam4 proxy: {point:+.6f}; exploratory paired 95% interval [{low:+.6f}, {high:+.6f}]')

# Additive full-evaluation contribution under fixed reference weights.
ww,cw=np.sqrt(baseline_eval.nwords),np.sqrt(baseline_eval.nchars)
paired['kenlm_proxy_gain']=.5*((baseline_eval.word_edits-chosen_eval.word_edits)*ww/ww.sum()/CONFIG['word_normalizer']+
    (baseline_eval.char_edits-chosen_eval.char_edits)*cw/cw.sum()/CONFIG['character_normalizer'])
assert np.isclose(paired.kenlm_proxy_gain.sum(),point)
display(paired.nlargest(12,'kenlm_proxy_gain'));display(paired.nsmallest(12,'kenlm_proxy_gain'))
paired.to_csv(OUT/'paired_evaluation_changes.csv',index=False)

decision={'checkpoint':SPEC,'calibration_size':len(calibration),'evaluation_size':len(evaluation),
    'selected_config':asdict(CHOSEN),'evaluation_beam4_proxy':summarize(baseline_eval)['zindi_proxy'],
    'evaluation_selected_proxy':summarize(chosen_eval)['zindi_proxy'],'evaluation_proxy_gain':float(point),
    'paired_bootstrap_95pct':[float(low),float(high)],'bootstrap_replicates':BOOTSTRAP_REPLICATES,
    'recommendation':('keep_beam4_baseline' if CHOSEN.fusion_lambda==0 or point<=0 or low<=0 else 'kenlm_supported_on_this_split'),
    'caveat':'No existing final selection or submission was changed.'}
write_json(OUT/'result.json',decision)
print(json.dumps(decision,indent=2));print('Reports:',OUT)

## Interpretation

- If calibration chooses λ = 0, KenLM failed its first gate; keep the existing decoder.
- If fusion wins calibration but not the 300-image evaluation, keep beam 4/greedy.
- If fusion improves evaluation but the paired interval includes zero, treat the gain as uncertain.
- Only a positive evaluation gain with a positive interval lower bound provides reasonably clear support on this split.

Even then, remember that greedy was 2.56× faster than beam 4 in the completed comparison and had essentially the same score. KenLM adds CPU-side scoring during every beam expansion, so it needs a useful accuracy gain—not merely the highest score by a tiny amount—to justify deployment.

In [ ]:
# RUN 3 — generate a resumable test submission with the calibration-selected decoder
assert 'CHOSEN' in globals(), 'Run the calibration and frozen-choice cells before creating a submission.'
assert CHOSEN.fusion_lambda > 0, 'Calibration selected visual-only decoding; use the existing beam-4 submission instead.'

test = pd.read_csv(ROOT / 'resources/Test.csv', dtype={'ID':str}, keep_default_na=False)
template = pd.read_csv(ROOT / 'resources/SampleSubmission.csv', dtype={'ID':str}, keep_default_na=False)
assert test.columns.tolist() == ['ID'] and test.ID.is_unique and len(test) > 0
assert template.columns.tolist() == ['ID','Target'] and template.ID.tolist() == test.ID.tolist()
assert all((IMAGES / f'{image_id}.jpg').is_file() for image_id in test.ID), 'A test image is missing.'

SUBMISSION_ROOT = OUT / f'test_submission_{CHOSEN.cache_key}'
SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)
submission_identity = {'validation_experiment_key':EXPERIMENT_KEY, 'checkpoint':SPEC,
    'selected_on':'calibration only', 'config':asdict(CHOSEN), 'test_ids_sha256':hashlib.sha256('\n'.join(test.ID).encode()).hexdigest()}
submission_manifest = SUBMISSION_ROOT / 'submission_experiment.json'
if submission_manifest.exists():
    assert json.loads(submission_manifest.read_text(encoding='utf-8')) == submission_identity, 'Submission settings changed; choose a new output directory.'
else:
    write_json(submission_manifest, submission_identity)

def test_cache_path(image_id):
    return SUBMISSION_ROOT / 'cache' / f'{image_id}.json'

def cached_test_prediction(image_id):
    path = test_cache_path(image_id)
    if not path.exists(): return None
    row = json.loads(path.read_text(encoding='utf-8'))
    assert row['ID'] == image_id and row['config'] == asdict(CHOSEN) and row['image_hash'] == file_hash(IMAGES / f'{image_id}.jpg')
    return row

pending = [image_id for image_id in test.ID if cached_test_prediction(image_id) is None]
print(f'Frozen test decoder: {CHOSEN}; cached: {len(test)-len(pending)}/{len(test)}')
if pending:
    model = load_model()
    try:
        warm = encode_image(pending[0]).to(DEVICE)
        decode(model, warm, CHOSEN)
        del warm
        for image_id in tqdm(pending, desc='test submission'):
            inputs = encode_image(image_id).to(DEVICE)
            try:
                result = decode(model, inputs, CHOSEN)
            except torch.cuda.OutOfMemoryError:
                print('GPU memory exhausted. Free memory and rerun; completed test predictions are cached.')
                raise
            write_json(test_cache_path(image_id), {'ID':image_id, 'image_hash':file_hash(IMAGES / f'{image_id}.jpg'),
                'config':asdict(CHOSEN), **result})
            del inputs
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()

predictions = [cached_test_prediction(image_id) for image_id in test.ID]
assert all(prediction is not None for prediction in predictions)
submission = pd.DataFrame({'ID':test.ID, 'Target':[prediction['Prediction'] for prediction in predictions]})
assert submission.ID.tolist() == template.ID.tolist() and submission.Target.notna().all()
submission_path = SUBMISSION_ROOT / f'submission_current_checkpoint_kenlm_{CHOSEN.cache_key}.csv'
submission.to_csv(submission_path, index=False)
write_json(SUBMISSION_ROOT / 'submission_metadata.json', {**submission_identity, 'rows':len(submission),
    'submission_file':submission_path.name, 'submission_sha256':file_hash(submission_path)})
display(submission.head()); print('Saved submission:', submission_path)
